### Structured Output

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

model=init_chat_model(
'groq:qwen/qwen3-32b'
)

In [18]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of movie")
    year:int=Field(description="Year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="Movie's rating out of 10")
    

In [19]:
model_with_structure=model.with_structured_output(Movie)

In [21]:
model_with_structure.invoke("Provide details about the movie Interstellar")

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

In [ ]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
model_with_structure.invoke("Provide details about the movie Interstellar")

MovieDetails(title='Interstellar', year=2014, cast=[Actor(name='Matthew McConaughey', role='Cooper'), Actor(name='Anne Hathaway', role='Amy'), Actor(name='Jessica Chastain', role='Murph')], genres=['Sci-Fi', 'Adventure', 'Drama'], budget=165.0)

### Typed Dict  

In [25]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """ A movie with details """
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"Year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"Movie's rating out of 10"]


model_with_typedDict=model.with_structured_output(MovieDict)
model_with_typedDict.invoke("Provide details about the movie Avengers Endgame")

{'director': 'Joe Russo, Anthony Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [26]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
model_with_structure.invoke("Provide details about the movie Interstellar")

{'budget': 165000000,
 'cast': [{'name': 'Matthew McConaughey', 'role': 'Cooper'},
  {'name': 'Anne Hathaway', 'role': 'Dr. Brand'},
  {'name': 'Jessica Chastain', 'role': 'Murf'}],
 'genres': ['Science Fiction', 'Adventure', 'Drama'],
 'title': 'Interstellar',
 'year': 2014}

### Data Classes

In [28]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information for a person"""
    name:str # The name of the person
    email:str # Email of the person
    phone:str  # Phone number of the person

agent=create_agent(
    model='groq:qwen/qwen3-32b',
    tools=[],
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from John Doe, john@example.com, 9123456780"
    }]
})
print(result['structured_response'])

ContactInfo(name='John Doe', email='john@example.com', phone='9123456780')
